# MedlinePlus Glossary — EDA


In [1]:
!pip install -q beautifulsoup4 requests

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

CATEGORY_URLS = {
    "Fitness": "https://medlineplus.gov/healthdefinitions/fitnessdefinitions.html",
    "General Health": "https://medlineplus.gov/healthdefinitions/generalhealthdefinitions.html",
    "Minerals": "https://medlineplus.gov/healthdefinitions/mineralsdefinitions.html",
    "Nutrition": "https://medlineplus.gov/healthdefinitions/nutritiondefinitions.html",
    "Vitamins": "https://medlineplus.gov/healthdefinitions/vitaminsdefinitions.html",
}

HEADERS = {"User-Agent": "Mozilla/5.0 (educational research use)"}

def parse_category(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    terms = []
    for h2 in soup.find_all("h2"):
        term = h2.get_text(strip=True)
        definition_parts = []
        source = None
        for sib in h2.find_next_siblings():
            if sib.name == "h2":
                break
            if sib.name == "p":
                text = sib.get_text(" ", strip=True)
                if text.lower().startswith("source"):
                    source = text
                else:
                    definition_parts.append(text)
        terms.append({"term": term, "definition": " ".join(definition_parts), "source": source})
    return terms

## 2. Parse all 5 categories into one table

In [4]:
rows = []
for category, url in CATEGORY_URLS.items():
    terms = parse_category(url)
    for t in terms:
        rows.append({"category": category, **t})
    time.sleep(1)

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(97, 4)


,category,term,definition,source
0,Fitness,Activity Count,Physical activity is any body movement that wo...,None
1,Fitness,Aerobic Exercise,Aerobic exercise is activity that moves your l...,None
2,Fitness,Basal Metabolic Rate,Basal metabolic rate is the measure of the ene...,None
3,Fitness,Body Mass Index,Body Mass Index (BMI) is an estimate of your b...,None
4,Fitness,Cool Down,Your physical activity session should end by g...,None


## 3. Terms per category

In [5]:
df["category"].value_counts()

,count
category,
Nutrition,30
General Health,18
Vitamins,17
Fitness,16
Minerals,16


## 4. Definition length

In [6]:
df["word_count"] = df["definition"].str.split().str.len()
df["word_count"].describe()

,word_count
count,97.000000
mean,51.896907
std,21.837222
min,15.000000
25%,37.000000
50%,51.000000
75%,61.000000
max,145.000000


## 5. Length by category

In [7]:
df.groupby("category")["word_count"].agg(["mean", "median", "min", "max"])

,mean,median,min,max
category,,,,
Fitness,43.250000,45.5,19,59
General Health,44.777778,47.5,15,86
Minerals,58.750000,58.5,30,98
Nutrition,53.200000,51.5,16,114
Vitamins,58.823529,56.0,31,145


## 6. Readability
Since this is meant to be *already* plain-language, this is your reference point for what patient-friendly reading level actually looks like — compare against the MTSamples baseline.

In [8]:
!pip install -q textstat
import textstat

def safe_fk(text):
    text = str(text)
    return textstat.flesch_kincaid_grade(text) if len(text.split()) >= 5 else None

df["flesch_kincaid_grade"] = df["definition"].apply(safe_fk)
df["flesch_kincaid_grade"].describe()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.0 MB/s eta 0:00:00


,flesch_kincaid_grade
count,97.000000
mean,8.258889
std,2.560451
min,3.229274
25%,6.462222
50%,8.163714
75%,10.148333
max,15.102449


## 7. Missing/empty definitions or sources

In [9]:
print("Empty definitions:", (df["definition"].str.strip() == "").sum())
print("Missing source attribution:", df["source"].isna().sum())

Empty definitions: 0
Missing source attribution: 97
